In [1]:
!pip install --quiet google-genai requests python-dotenv

**Solar Optimization Advisor - Multi-Agent System**

The **Solar Advisor Agent** is an intelligent system designed to help users determine the optimal solar panel setup for their home or building. It combines **real-time weather analysis**, **solar irradiance estimation**, and **engineering-based calculations** with **LLM-driven advisory explanations**.

Demonstrates: Sequential agent workflow, structured data extraction,
memory management, and tool integration patterns.

Architecture:
- InputExtractorAgent: Parses user prompt to extract parameters
- DataCollectorAgent: Fetches weather and solar irradiance data
- ComputationAgent: Calculates panel layout, energy yield, and ROI
- AdvisorAgent: Orchestrates workflow and generates final recommendation

Key Features:
- Single-prompt interaction (user provides all details at once)
- Stateful session management with memory compression
- External API integration (OpenWeather, Open-Meteo)
- Cost-benefit analysis with configurable parameters

In [2]:
import os
import json
import re
import logging
from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional, List

import requests
from dotenv import load_dotenv
from google import genai

load_dotenv()

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()


# API credentials
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")
WEATHER_API_KEY = user_secrets.get_secret("WEATHER_API_KEY")

# External API endpoints
WEATHER_API_URL = "https://api.openweathermap.org/data/2.5/weather"
SOLAR_API_BASE_URL = "https://api.open-meteo.com/v1/forecast"

# Solar panel specifications (industry standard 400W panel)
PANEL_LENGTH_M = 1.7        # Standard panel length in meters
PANEL_WIDTH_M = 1.1         # Standard panel width in meters
PANEL_AREA_M2 = PANEL_LENGTH_M * PANEL_WIDTH_M  # Total area per panel
PANEL_WATT = 400            # Rated power output per panel
PANEL_COST_USD = 200        # Cost per panel

# System parameters
BOS_MULTIPLIER = 1.3        # Balance of System cost multiplier (inverter, wiring, installation)
DERATE_FACTOR = 0.8         # System efficiency factor (accounts for losses: temperature, wiring, inverter, etc.)


# Configure Gemini client
client = genai.Client(api_key=GOOGLE_API_KEY)

# Logging configuration for observability and debugging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("solar-advisor")

**SolarSessionState Dataclass + In-memory Session Service**

Defines a SolarSessionState dataclass to store:

- user location

- roof dimensions

- tilt/azimuth

- budget

- session ID

Implements InMemorySessionService to simulate session persistence.

Helps maintain user input across multiple agent calls.

In [3]:
@dataclass
class SolarSessionState:
    
    user_location: Optional[str] = None
    roof_length_m: Optional[float] = None
    roof_width_m: Optional[float] = None
    tilt_deg: Optional[float] = None
    azimuth_deg: Optional[float] = None
    budget_usd: Optional[float] = None
    monthly_units_kwh: Optional[float] = None
    last_results: Optional[Dict[str, Any]] = None
    summary: Optional[str] = None


class InMemorySessionService:
   

    def __init__(self):
        self._sessions: Dict[str, SolarSessionState] = {}

    def get_or_create(self, session_id: str) -> SolarSessionState:
        """Retrieve existing session or create new one."""
        if session_id not in self._sessions:
            logger.info(f"Creating new session: {session_id}")
            self._sessions[session_id] = SolarSessionState()
        return self._sessions[session_id]

    def update_summary(self, session_id: str, summary: str):
        """Update compressed memory summary to avoid context bloat."""
        state = self.get_or_create(session_id)
        state.summary = summary
        logger.info(f"Updated summary for session={session_id}")


# Global session service instance
session_service = InMemorySessionService()

**Fetch Weather**

This function calls the OpenWeatherMap API using a city name, retrieves the temperature, cloud cover, and overall weather conditions, logs both the request and response, and returns a weather dictionary for use by the advisor agent.

In [4]:
def fetch_weather(location: str) -> Dict[str, Any]:
    """
    Fetch current weather data for a given location.

    Uses OpenWeather API to get:
    - Temperature
    - Cloud cover (affects solar production)
    - Geographic coordinates (lat/lon for solar irradiance lookup)

    Args:
        location: City name (e.g., "Delhi,IN" or "New York,US")

    Returns:
        Dict containing weather data and coordinates

    Raises:
        requests.HTTPError: If API request fails
    """
    logger.info(f"[TOOL] fetch_weather called: {location}")

    params = {
        "q": location,
        "appid": WEATHER_API_KEY,
        "units": "metric",  # Use Celsius
    }

    resp = requests.get(WEATHER_API_URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    result = {
        "location": location,
        "temperature_c": data["main"]["temp"],
        "cloud_cover_percent": data.get("clouds", {}).get("all", 0),
        "description": data["weather"][0]["description"],
        "raw": data  # Keep full response for coordinate extraction
    }

    logger.info(f"[TOOL] fetch_weather result: {result}")
    return result

**Fetch Solar Irradiance**

This function retrieves hourly solar irradiance data from the Solcast or Open-Meteo API using latitude and longitude. It logs the entire fetch process and returns the solar energy potential needed for modeling calculations.

In [5]:
def fetch_solar_irradiance(lat: float, lon: float) -> Dict[str, Any]:
    """
    Fetch solar irradiance data for specific coordinates.

    Uses Open-Meteo API to get hourly shortwave radiation (W/m²).
    Converts to daily kWh/m²/day for energy calculations.

    Global Horizontal Irradiance (GHI) is the total solar radiation
    received on a horizontal surface, key for solar potential.

    Args:
        lat: Latitude in decimal degrees
        lon: Longitude in decimal degrees

    Returns:
        Dict containing daily GHI in kWh/m²/day
    """
    logger.info(f"[TOOL] fetch_solar_irradiance called lat={lat}, lon={lon}")

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "shortwave_radiation"  # Solar radiation in W/m²
    }

    resp = requests.get(SOLAR_API_BASE_URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    radiation = data.get("hourly", {}).get("shortwave_radiation", [])

    if not radiation:
        return {"error": "No solar data found from Open-Meteo"}

    # Convert hourly W/m² to daily kWh/m²
    # Formula: Sum(W/m²) * 0.001 (W to kW conversion)
    daily_kwh = sum(radiation) * 0.001

    result = {
        "ghi_kwh_m2_day": round(daily_kwh, 3),
        "source": "open-meteo"
    }

    logger.info(f"[TOOL] fetch_solar_irradiance result: {result}")
    return result

**Search Electricity Rate** 

This function searches the web for residential electricity rates at a specified location. It uses Gemini with Google Search to obtain the current cost per kWh, then parses the results with an LLM to extract the numerical rate. This enables real-time, location-specific pricing instead of relying on hardcoded values.

In [6]:
def search_electricity_rate(location: str) -> Dict[str, Any]:
    
    logger.info(f"[TOOL] search_electricity_rate called: {location}")

    # Search query optimized for finding electricity rates
    search_query = f"What is the average residential electricity rate cost per kWh in {location} in 2024? Include utility company rates."

    try:
        # Use Gemini with Google Search grounding
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=search_query,
            config={
                "tools": [{"google_search": {}}]
            }
        )

        # Extract rate from search results using structured prompt
        extraction_prompt = f"""
        Based on the search results about electricity rates in {location},
        extract the residential electricity cost per kWh in USD.

        Return ONLY a JSON object with this exact format:
        {{"electricity_rate_usd_per_kwh": <number>, "source": "<utility or source name>"}}

        Search results: {response.text}

        If you cannot find a specific rate, use 0.12 as default and mention "estimated average".
        """

        rate_response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=extraction_prompt
        )

        # Parse JSON response
        text = rate_response.text.strip()
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()

        result = json.loads(text)
        logger.info(f"[TOOL] Electricity rate found: {result}")
        return result

    except Exception as e:
        # Fallback to US average if search fails
        logger.warning(f"Failed to search electricity rate: {e}. Using default.")
        return {
            "electricity_rate_usd_per_kwh": 0.12,
            "source": "US national average (search unavailable)"
        }


**Calculate optimal panel arrangement on roof.**

Algorithm:

    1. Divide roof dimensions by panel dimensions
    
    2. Use floor division to get integer panel count
    
    3. Calculate area utilization

Assumes portrait orientation (1.7m x 1.1m per panel).

Does not account for:

    * Obstructions (chimneys, vents)
    
    * Edge spacing requirements
    
    * Fire code setbacks

In [7]:
def compute_panel_layout(roof_length_m: float, roof_width_m: float) -> Dict[str, Any]:
    
    logger.info("[TOOL] compute_panel_layout called")

    # Calculate how many panels fit along each dimension
    panels_length = int(roof_length_m // PANEL_LENGTH_M)
    panels_width = int(roof_width_m // PANEL_WIDTH_M)
    total = panels_length * panels_width

    # Calculate area metrics
    used_area = total * PANEL_AREA_M2
    roof_area = roof_length_m * roof_width_m

    layout = {
        "panels_along_length": panels_length,
        "panels_along_width": panels_width,
        "total_panels": total,
        "roof_area_m2": roof_area,
        "used_area_m2": used_area,
        "utilization": used_area / roof_area if roof_area > 0 else 0
    }

    logger.info(f"[TOOL] Panel layout: {layout}")
    return layout

Estimate annual energy production considering orientation factors.

    Calculation steps:
    1. Calculate total panel area
    2. Apply tilt efficiency factor (optimal: 30°)
    3. Apply azimuth efficiency factor (optimal: 180° = south)
    4. Apply system derate factor (losses)
    5. Scale to yearly production

    Tilt factor: Linear penalty for deviation from 30°
    Azimuth factor: 180° (south) is optimal in Northern Hemisphere

In [8]:
def estimate_energy_yield(total_panels: int, ghi: float, tilt: float, azimuth: float) -> Dict[str, Any]:
    
    logger.info("[TOOL] estimate_energy_yield called")

    area = total_panels * PANEL_AREA_M2

    # Tilt efficiency: Optimal at 30°, penalty for deviation
    # Factor ranges from 0.7 to 1.0
    tilt_factor = 1.0 - abs(tilt - 30) / 100
    tilt_factor = max(0.7, min(1.0, tilt_factor))

    # Azimuth efficiency: 180° (south) is optimal
    # Calculate shortest angular distance to 180°
    az_diff = min(abs(azimuth - 180), abs(azimuth + 180))
    az_factor = 1.0 - az_diff / 360
    az_factor = max(0.7, min(1.0, az_factor))

    # Combined orientation factor
    orientation = tilt_factor * az_factor

    # Energy calculation: GHI * Area * Derate * Orientation
    daily_kwh = ghi * area * DERATE_FACTOR * orientation
    yearly_kwh = daily_kwh * 365

    result = {
        "area_m2": area,
        "orientation_factor": orientation,
        "daily_kwh": daily_kwh,
        "yearly_kwh": yearly_kwh
    }

    logger.info(f"[TOOL] energy yield: {result}")
    return result


**Estimate annual energy production considering orientation factors.**

    Calculation steps:
    1. Calculate total panel area
    2. Apply tilt efficiency factor (optimal: 30°)
    3. Apply azimuth efficiency factor (optimal: 180° = south)
    4. Apply system derate factor (losses)
    5. Scale to yearly production

    Tilt factor: Linear penalty for deviation from 30°
    Azimuth factor: 180° (south) is optimal in Northern Hemisphere

In [9]:
def estimate_energy_yield(total_panels: int, ghi: float, tilt: float, azimuth: float) -> Dict[str, Any]:
    
    logger.info("[TOOL] estimate_energy_yield called")

    area = total_panels * PANEL_AREA_M2

    # Tilt efficiency: Optimal at 30°, penalty for deviation
    # Factor ranges from 0.7 to 1.0
    tilt_factor = 1.0 - abs(tilt - 30) / 100
    tilt_factor = max(0.7, min(1.0, tilt_factor))

    # Azimuth efficiency: 180° (south) is optimal
    # Calculate shortest angular distance to 180°
    az_diff = min(abs(azimuth - 180), abs(azimuth + 180))
    az_factor = 1.0 - az_diff / 360
    az_factor = max(0.7, min(1.0, az_factor))

    # Combined orientation factor
    orientation = tilt_factor * az_factor

    # Energy calculation: GHI * Area * Derate * Orientation
    daily_kwh = ghi * area * DERATE_FACTOR * orientation
    yearly_kwh = daily_kwh * 365

    result = {
        "area_m2": area,
        "orientation_factor": orientation,
        "daily_kwh": daily_kwh,
        "yearly_kwh": yearly_kwh
    }

    logger.info(f"[TOOL] energy yield: {result}")
    return result

**Calculate system cost and financial return on investment.**

Cost components:

 * Panel cost: Number of panels * unit cost
 
 * BOS (Balance of System): Inverter, wiring, mounting, installation

ROI calculation:

 * Annual savings = Production * Electricity rate (now dynamic per location)
 
 * Payback period = Total cost / Annual savings

In [10]:
def estimate_cost_and_roi(total_panels: int, yearly_kwh: float, budget: Optional[float],
                          electricity_rate: float) -> Dict[str, Any]:
    
    logger.info("[TOOL] estimate_cost_and_roi called")

    # Calculate total system cost
    panel_cost = total_panels * PANEL_COST_USD
    total_cost = panel_cost * BOS_MULTIPLIER  # Add BOS costs

    # Calculate financial returns using dynamic electricity rate
    annual_savings = yearly_kwh * electricity_rate
    payback = total_cost / annual_savings if annual_savings else None

    # Check budget constraint if provided
    within_budget = None
    if budget is not None:
        within_budget = total_cost <= budget

    result = {
        "system_cost_usd": total_cost,
        "annual_savings_usd": annual_savings,
        "electricity_rate_used": electricity_rate,  # Show rate used for transparency
        "payback_years": payback,
        "within_budget": within_budget
    }

    logger.info(f"[TOOL] ROI: {result}")
    return result


**ComputationAgent**

Orchestrates the calculation workflow:

 1. Panel layout optimization
    
 2. Energy yield estimation
    
 3. Cost and ROI analysis
    
This agent is called by the main AdvisorAgent after data collection.

In [11]:
class ComputationAgent:
   

    def run(self, L: float, W: float, tilt: float, azimuth: float,
            ghi: float, budget: Optional[float], electricity_rate: float) -> Dict[str, Any]:
        """
        Execute full computation pipeline.

        Args:
            electricity_rate: Dynamic rate from web search ($/kWh)

        Returns error if roof is too small, otherwise returns
        complete analysis with layout, energy, and ROI data.
        """
        layout = compute_panel_layout(L, W)

        # Early return if no panels fit
        if layout["total_panels"] == 0:
            return {"error": "Roof too small", "layout": layout}

        # Continue with energy and financial calculations
        energy = estimate_energy_yield(layout["total_panels"], ghi, tilt, azimuth)
        roi = estimate_cost_and_roi(layout["total_panels"], energy["yearly_kwh"],
                                     budget, electricity_rate)

        return {"layout": layout, "energy": energy, "roi": roi}


# Global computation agent instance
computation_agent = ComputationAgent()

Compress previous results into concise summary.

In [12]:
def compact_context(state: SolarSessionState) -> str:
    
    if not state.last_results:
        return "No previous runs."

    prompt = f"""
    Summarize the following solar PV results into 3-5 technical bullet points:

    {json.dumps(state.last_results, indent=2)}
    """

    resp = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )

    return resp.text.strip()


LLM-based agent for parsing user prompts and extracting parameters.


In [13]:
class InputExtractorAgent:


    def __init__(self):
        self.model = "gemini-2.5-flash-lite"

    def extract(self, user_prompt: str) -> Dict[str, Any]:
       
        logger.info("[AGENT] InputExtractorAgent: Parsing user prompt")

        extraction_prompt = f"""
        Extract solar panel parameters from the user's request.

        USER REQUEST:
        {user_prompt}

        Extract and return JSON with these fields:
        - location (city name, e.g., "Delhi,IN" or "New York,US")
        - roof_length_m (number)
        - roof_width_m (number)
        - tilt_deg (number, 0-90)
        - azimuth_deg (number, 0=North, 90=East, 180=South, 270=West)
        - budget_usd (number or null)
        - monthly_units_kwh (number or null - user's monthly electricity consumption)

        If orientation is described as "south-facing", use azimuth=180.
        If "north-facing", use azimuth=0. If "east-facing", use azimuth=90. If "west-facing", use azimuth=270.
        If information is missing, use null.

        Return ONLY valid JSON, no explanation.
        """

        resp = client.models.generate_content(
            model=self.model,
            contents=extraction_prompt
        )

        # Parse LLM response as JSON
        try:
            # Extract JSON from response (handle markdown code blocks)
            text = resp.text.strip()
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0].strip()
            elif "```" in text:
                text = text.split("```")[1].split("```")[0].strip()

            extracted = json.loads(text)
            logger.info(f"[AGENT] Extracted parameters: {extracted}")
            return extracted
        except json.JSONDecodeError as e:
            logger.error(f"Failed to parse extraction: {e}")
            return {}

**Main orchestrator agent - coordinates the full workflow.**

    Workflow:
    1. Extract parameters from user prompt (InputExtractorAgent)
    2. Fetch weather and solar data (DataCollectorAgent pattern)
    3. Search for local electricity rates (ElectricityRateAgent pattern)
    4. Run calculations (ComputationAgent with dynamic pricing)
    5. Generate human-friendly explanation (LLM synthesis)
    6. Update session memory

    This agent demonstrates the "orchestrator + specialist" pattern
    common in multi-agent systems.

In [14]:
class AdvisorAgent:
   

    def __init__(self):
        self.model = "gemini-2.5-flash-lite"
        self.input_extractor = InputExtractorAgent()

    def advise(self, session_id: str, user_prompt: str) -> str:
        
        logger.info(f"[AGENT] AdvisorAgent: Processing request for session {session_id}")

        # Step 1: Extract parameters from natural language prompt
        params = self.input_extractor.extract(user_prompt)

        if not params or not params.get("location"):
            return "I couldn't extract the required information. Please provide: location, roof dimensions (length x width in meters), tilt angle, and azimuth direction."

        # Extract parameters with defaults
        location = params["location"]
        L = params.get("roof_length_m")
        W = params.get("roof_width_m")
        tilt = params.get("tilt_deg", 30)  # Default tilt
        azimuth = params.get("azimuth_deg", 180)  # Default to south
        budget = params.get("budget_usd")
        monthly_units = params.get("monthly_units_kwh")

        # Validate required parameters
        if not L or not W:
            return "I need roof dimensions (length and width in meters) to proceed."

        # Get or create session state
        state = session_service.get_or_create(session_id)

        # Store inputs in session
        state.user_location = location
        state.roof_length_m = L
        state.roof_width_m = W
        state.tilt_deg = tilt
        state.azimuth_deg = azimuth
        state.budget_usd = budget
        state.monthly_units_kwh = monthly_units

        # Compress previous context for memory efficiency
        compact = compact_context(state)

        # Step 2: Collect external data
        logger.info("[AGENT] DataCollector: Fetching weather and irradiance data")

        try:
            weather = fetch_weather(location)
            lat = weather["raw"]["coord"]["lat"]
            lon = weather["raw"]["coord"]["lon"]
            irradiance = fetch_solar_irradiance(lat, lon)
            ghi = irradiance["ghi_kwh_m2_day"]
        except Exception as e:
            logger.error(f"Data collection failed: {e}")
            return f"Failed to fetch weather/solar data for {location}. Please check the location name."

        # Step 3: Search for electricity rate (NEW!)
        logger.info("[AGENT] ElectricityRateAgent: Searching for local electricity rates")
        electricity_data = search_electricity_rate(location)
        electricity_rate = electricity_data["electricity_rate_usd_per_kwh"]

        # Step 4: Run computations with dynamic electricity rate
        logger.info("[AGENT] ComputationAgent: Calculating layout, energy, and ROI")
        calc = computation_agent.run(L, W, tilt, azimuth, ghi, budget, electricity_rate)

        # Check for computation errors
        if "error" in calc:
            return f"Analysis failed: {calc['error']}"

        # Step 5: Save results to session
        state.last_results = {
            "inputs": {
                "location": location,
                "roof_length_m": L,
                "roof_width_m": W,
                "tilt_deg": tilt,
                "azimuth_deg": azimuth,
                "budget_usd": budget,
                "monthly_units_kwh": monthly_units
            },
            "weather": weather,
            "irradiance": irradiance,
            "electricity_rate": electricity_data,  # Include rate data
            "calculations": calc
        }

        # Step 6: Generate natural language explanation using LLM
        logger.info("[AGENT] ReportGenerator: Creating user-friendly explanation")

        explanation_prompt = f"""
        You are a Solar Advisor. Using the JSON below, produce a clear, friendly explanation covering:

        - Panel layout summary (how many panels fit, utilization)
        - Yearly energy estimate
        - System cost and payback period
        - Electricity rate used for calculations (mention the source: {electricity_data['source']})
        - If monthly consumption was provided, compare solar production with their usage
        - Budget assessment (if budget was provided)
        - Overall recommendation

        PREVIOUS CONTEXT:
        {compact}

        CURRENT ANALYSIS:
        {json.dumps(state.last_results, indent=2)}

        Be conversational but informative. Use specific numbers from the data.
        Highlight that electricity rates were searched specifically for their location.
        """

        resp = client.models.generate_content(
            model=self.model,
            contents=explanation_prompt
        )

        explanation = resp.text.strip()

        # Step 7: Update session memory with compressed summary
        logger.info("[AGENT] Updating session memory")

        summary_prompt = f"""
        Create a compact 3-5 bullet technical summary from:
        {json.dumps(state.last_results, indent=2)}
        """

        summary_resp = client.models.generate_content(
            model=self.model,
            contents=summary_prompt
        )

        session_service.update_summary(session_id, summary_resp.text.strip())

        return explanation


# Global advisor agent instance
advisor_agent = AdvisorAgent()


In [15]:
import logging
logging.disable(logging.CRITICAL)


In [17]:
def run_solar_advisor_interaction():
   
    print("=== Solar Optimization Advisor (Agent Mode) ===\n")
    print("Provide all details in one prompt: location, roof size in meters (LxW), tilt in degrees, direction (N/S/E/W), budget in USD, monthly electricity usage in kWh.\n")
    print("Example: 'I'm in Delhi, India with a 10x6m roof, 25° tilt, south-facing, $5000 budget, 600 kWh/month'\n")
    

    session_id = "demo-user-1"

    user_prompt = input("Your request: ")

    if not user_prompt.strip():
        print("No input provided.")
        return

    print("\n--- Analyzing (including web search for electricity rates)... ---\n")

    result = advisor_agent.advise(session_id, user_prompt)

    print("\n--- Solar Advisor Response ---\n")
    print(result)

import warnings
warnings.filterwarnings('ignore')

# Run the advisor
if __name__ == "__main__":
    run_solar_advisor_interaction()

=== Solar Optimization Advisor (Agent Mode) ===

Provide all details in one prompt: location, roof size in meters (LxW), tilt in degrees, direction (N/S/E/W), budget in USD, monthly electricity usage in kWh.

Example: 'I'm in Delhi, India with a 10x6m roof, 25° tilt, south-facing, $5000 budget, 600 kWh/month'



Your request:  I'm in Delhi, India with a 10x6m roof, 25° tilt, south-facing, $5000 budget, 600 kWh/month



--- Analyzing (including web search for electricity rates)... ---


--- Solar Advisor Response ---

Hello there! I'm happy to walk you through the exciting possibilities of solar energy for your home. Based on the information we have, here's a clear picture of what a solar system could look like for you:

### Your Solar Potential at a Glance

*   **Panel Layout:** We can fit a robust system with **25 solar panels** on your roof. This utilizes **46.75 square meters** of your available **60 square meter** roof space, giving us a great **77.9% roof utilization**. This means we're making excellent use of the sunny areas!

*   **Estimated Yearly Energy Production:** This system is projected to generate a fantastic amount of clean energy, with an estimated **363,208 kWh per year**. That's a significant contribution to your home's power needs!

*   **System Cost and Payback:** The estimated cost for this 25-panel system is **$6,500**. Now, let's talk about savings. With an electricity rate o